In [18]:
import gradio as gr
from pypdf import PdfReader
from IPython.display import display, Markdown
from openai import OpenAI

In [19]:
def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

In [20]:
extracted_text = extract_text_from_pdf("documents/linkedin.pdf")

system_prompt = f"""
You are digital twin of the person whose LinkedIn profile is provided.
You have to answer questions about the person based on the information provided in the LinkedIn profile.

Here is the LinkedIn profile information:
{extracted_text}

Key points to remember:
- You should answer questions based on the information provided in the LinkedIn profile.
- If the information is not available in the LinkedIn profile, you should respond with "I don't know" or "I don't have that information".
- Never invlolve in any personal opinions or information that is not present in the LinkedIn profile.
"""

In [21]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1/"
)

In [22]:
def chat_with_ollama(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    reply = client.chat.completions.create(
        model="ministral-3:8b",
        messages=messages
    )
    return reply.choices[0].message.content

In [25]:
chat_interface = gr.ChatInterface(
    fn=chat_with_ollama, title="Chat with Talant's Digital Twin"    
    )

# chat_interface.launch(inline=True, share=True, inbrowser=True)